In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import *

from sklearn.model_selection import GridSearchCV

In [2]:
!pip install mlflow albumentations -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import mlflow
import mlflow.pytorch

In [4]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [5]:
# Clonar el repo
if not os.path.exists("skin-dataset-classification"):
    !git clone https://github.com/2245-RN-ITBA/skin-dataset-classification.git

# Moverse al directorio del repo
os.chdir("skin-dataset-classification")

# Verificar que el dataset está
print(os.listdir("data/Split_smol"))

Cloning into 'skin-dataset-classification'...
remote: Enumerating objects: 934, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 934 (delta 17), reused 15 (delta 15), pack-reused 911 (from 2)
Receiving objects: 100% (934/934), 222.43 MiB | 22.39 MiB/s, done.
Resolving deltas: 100% (31/31), done.
['val', 'train']


In [6]:
mlflow.set_tracking_uri("sqlite:///mlflow_2.db")
mlflow.set_experiment("Clasificador_Imagenes")

2026/06/20 17:30:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/20 17:30:04 INFO mlflow.store.db.utils: Updating database tables
2026/06/20 17:30:06 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Imagenes' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/skin-dataset-classification/mlruns/1', creation_time=1781976606020, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781976606020, lifecycle_stage='active', name='Clasificador_Imagenes', tags={}, trace_location=None, workspace='default'>

## Funciones auxiliares

Se definen acá todas las funciones que en versiones anteriores venían de `helper.py`,
para que el notebook sea autocontenido.

In [7]:
# NUEVA

# Convierte una figura matplotlib a imagen y la loguea en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)


# Cuenta los parámetros entrenables del modelo
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [8]:
# MODIFICADA

def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=classes,
                                       zero_division=0)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)

In [9]:
# Entrenamiento y validación
def evaluate(model, loader, writer, device, classes, criterion, epoch=None, prefix="val"):
    log_classification_report(model, loader, writer, device, classes, step=epoch , prefix="val")
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

## Dataset y Modelo


In [10]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels      = []

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)

        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return image, label

In [11]:
class MLPClassifier(nn.Module):

    def __init__(self, input_size=64*64*3, num_classes=10, dropout=0.0):
        super().__init__()

        layers = [
            nn.Flatten(),
            nn.Linear(input_size, 512),
            nn.ReLU(),
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(p=dropout))

        layers += [
            nn.Linear(512, 128),
            nn.ReLU(),
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(p=dropout))

        layers.append(nn.Linear(128, num_classes))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Espacio de búsqueda de hiperparámetros

Se define el espacio de HPs a explorar. El producto cartesiano de todas las listas
daría `3×3×3×2×2×2×2×4 = 1728` combinaciones. Como entrenar todas sería muy costoso,
se muestrea aleatoriamente ~5% de ellas (`np.random.rand() < 0.05`).

Los HPs explorados son:

| HP | Valores |
|---|---|
| `input_size` (resolución imagen) | 32, 64, 128 |
| `batch_size` | 16, 64, 128 |
| `lr` | 1e-2, 1e-3, 1e-4 |
| `optimizer` | Adam, SGD |
| `HFlip` (prob. flip horizontal) | 0.0, 0.5 |
| `VFlip` (prob. flip vertical) | 0.0, 0.5 |
| `RBContrast` (prob. brightness/contrast) | 0.0, 0.5 |
| `dropout` | 0.0, 0.1, 0.2, 0.3 |

In [12]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [13]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_skin"
writer = SummaryWriter(log_dir=log_dir)

In [14]:
np.random.rand()

0.3070152086793584

In [15]:
# reduje los hiperparametros para que corra

hparams_space = {
    "model":       "MLPClassifier",
    "input_size":  [32, 64],
    "batch_size":  [16, 64],
    "lr":          [1e-2, 1e-3, 1e-4],
    "epochs":      50,
    "optimizer":   ["Adam", "SGD"],
    "HFlip":       [0.0, 0.5],
    "VFlip":       [0.0, 0.5],
    "RBContrast":  [0.0, 0.5],
    "loss_fn":     "CrossEntropyLoss",
    "train_dir":   train_dir,
    "val_dir":     val_dir,
    "es_patience": 5,
    "dropout":     [0.0, 0.2],
}

## Loop de búsqueda aleatoria

Para cada combinación sorteada:
1. Se construyen los datasets y dataloaders con el `input_size` y augmentations correspondientes.
2. Se instancia el modelo con el `dropout` elegido.
3. Se entrena hasta 200 épocas con **Early Stopping** (paciencia de 5 épocas sin mejora en val_acc).
4. Se loguean en MLflow los HPs y las mejores métricas de cada run.

In [16]:
np.random.seed(42)

modelnbr = 0
for input_size in hparams_space["input_size"]:
    for bs in hparams_space["batch_size"]:
        for lr in hparams_space["lr"]:
            for optimizer_name in hparams_space["optimizer"]:
                for HFlip in hparams_space["HFlip"]:
                    for VFlip in hparams_space["VFlip"]:
                        for RBContrast in hparams_space["RBContrast"]:
                            for dropout in hparams_space["dropout"]:
                                if np.random.rand() < 0.05:
                                    modelnbr += 1
                                    print(f"\n=== Modelo #{modelnbr} | input={input_size} bs={bs} lr={lr} opt={optimizer_name} dropout={dropout} ===")


                                    hparams = {
                                        "model":       "MLPClassifier",
                                        "input_size":  input_size,
                                        "batch_size":  bs,
                                        "lr":          lr,
                                        "epochs":      50,
                                        "optimizer":   optimizer_name,
                                        "HFlip":       HFlip,
                                        "VFlip":       VFlip,
                                        "RBContrast":  RBContrast,
                                        "loss_fn":     "CrossEntropyLoss",
                                        "train_dir":   train_dir,
                                        "val_dir":     val_dir,
                                        "es_patience": 5,
                                        "dropout":     dropout,
                                    }

                                    train_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.HorizontalFlip(p=hparams["HFlip"]),
                                        A.VerticalFlip(p=hparams["VFlip"]),
                                        A.RandomBrightnessContrast(p=hparams["RBContrast"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])
                                    val_test_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])

                                    train_dataset = CustomImageDataset(train_dir, transform=train_transform)
                                    val_dataset   = CustomImageDataset(val_dir,   transform=val_test_transform)
                                    t_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
                                    v_loader = DataLoader(val_dataset,   batch_size=bs)

                                    device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                                    num_classes = len(set(train_dataset.labels))
                                    model = MLPClassifier(
                                        num_classes=num_classes,
                                        input_size=hparams["input_size"]**2 * 3,
                                        dropout=hparams["dropout"]
                                    ).to(device)

                                    criterion = nn.CrossEntropyLoss()
                                    if optimizer_name == "Adam":
                                        optimizer = optim.Adam(model.parameters(), lr=hparams["lr"])
                                    else:
                                        optimizer = optim.SGD(model.parameters(), lr=hparams["lr"], momentum=0.9)

                                    hparams["count_params"] = count_parameters(model)

                                    with mlflow.start_run():
                                        mlflow.log_params(hparams)

                                        best_val_acc    = 0
                                        best_val_loss   = 0
                                        best_train_acc  = 0
                                        best_train_loss = 0
                                        best_epoch      = 0

                                        for epoch in range(hparams["epochs"]):
                                            model.train()
                                            running_loss   = 0.0
                                            correct, total = 0, 0

                                            for images, labels in t_loader:
                                                images, labels = images.to(device), labels.to(device)
                                                optimizer.zero_grad()
                                                outputs = model(images)
                                                loss    = criterion(outputs, labels)
                                                loss.backward()
                                                optimizer.step()
                                                running_loss += loss.item()
                                                _, preds = torch.max(outputs, 1)
                                                correct  += (preds == labels).sum().item()
                                                total    += labels.size(0)

                                            train_loss = running_loss / len(t_loader)
                                            train_acc  = 100.0 * correct / total
                                            val_loss, val_acc = evaluate(
                                                model, v_loader, writer, device,
                                                train_dataset.label_encoder.classes_,
                                                criterion,
                                                epoch=epoch,
                                                prefix="val"
                                            )

                                            print(f"  Epoch {epoch+1:3d}/{hparams['epochs']} | "
                                                  f"train_loss={train_loss:.4f} train_acc={train_acc:.1f}% | "
                                                  f"val_loss={val_loss:.4f} val_acc={val_acc:.1f}%")

                                            writer.add_scalar("train/loss",     train_loss, epoch)
                                            writer.add_scalar("train/accuracy", train_acc,  epoch)

                                            mlflow.log_metrics({
                                                "train_loss":     train_loss,
                                                "train_accuracy": train_acc,
                                                "val_loss":       val_loss,
                                                "val_accuracy":   val_acc
                                            }, step=epoch)

                                            if val_acc > best_val_acc:
                                                best_val_acc    = val_acc
                                                best_val_loss   = val_loss
                                                best_train_acc  = train_acc
                                                best_train_loss = train_loss
                                                best_epoch      = epoch
                                                torch.save(model.state_dict(), "mlp_model.pth")
                                                mlflow.log_artifact("mlp_model.pth")
                                            elif epoch > best_epoch + hparams["es_patience"]:
                                                print(f"  Early stopping en epoch {epoch+1}")
                                                break

                                        mlflow.log_metrics({
                                            "best_train_loss":     best_train_loss,
                                            "best_train_accuracy": best_train_acc,
                                            "best_val_loss":       best_val_loss,
                                            "best_val_accuracy":   best_val_acc,
                                            "best_epoch":          best_epoch
                                        }, step=epoch + 1)

print(f"\nBúsqueda finalizada. Se entrenaron {modelnbr} modelos.")


=== Modelo #1 | input=32 bs=16 lr=0.01 opt=Adam dropout=0.0 ===
  Epoch   1/50 | train_loss=6.2196 train_acc=15.8% | val_loss=2.0968 val_acc=17.1%
  Epoch   2/50 | train_loss=2.1488 train_acc=14.1% | val_loss=2.0904 val_acc=18.8%
  Epoch   3/50 | train_loss=2.2114 train_acc=12.8% | val_loss=2.1710 val_acc=13.3%
  Epoch   4/50 | train_loss=2.1605 train_acc=12.1% | val_loss=2.0932 val_acc=16.0%
  Epoch   5/50 | train_loss=2.1567 train_acc=13.1% | val_loss=2.1632 val_acc=13.3%
  Epoch   6/50 | train_loss=2.1766 train_acc=13.3% | val_loss=2.1806 val_acc=12.2%
  Epoch   7/50 | train_loss=2.1957 train_acc=15.5% | val_loss=2.1615 val_acc=13.8%
  Epoch   8/50 | train_loss=2.2795 train_acc=11.3% | val_loss=2.2033 val_acc=11.0%
  Early stopping en epoch 8

=== Modelo #2 | input=32 bs=16 lr=0.01 opt=SGD dropout=0.2 ===
  Epoch   1/50 | train_loss=1.9353 train_acc=27.5% | val_loss=1.5735 val_acc=43.6%
  Epoch   2/50 | train_loss=1.6618 train_acc=37.3% | val_loss=1.5060 val_acc=37.0%
  Epoch   3/5

In [17]:
import pandas as pd

mlflow.set_tracking_uri("sqlite:///mlflow_2.db")
runs = mlflow.search_runs(
    experiment_names=["Clasificador_Imagenes"],
    order_by=["metrics.best_val_accuracy DESC"]
)

cols = ["params.model", "params.input_size", "params.batch_size", "params.lr",
        "params.optimizer", "params.dropout", "params.HFlip", "params.VFlip",
        "params.RBContrast", "metrics.best_val_accuracy", "metrics.best_train_accuracy",
        "metrics.best_epoch"]
cols = [c for c in cols if c in runs.columns]

print(f"Total de runs: {len(runs)}")
runs[cols].head(10)

Total de runs: 26


,params.model,params.input_size,params.batch_size,params.lr,params.optimizer,params.dropout,params.HFlip,params.VFlip,params.RBContrast,metrics.best_val_accuracy,metrics.best_train_accuracy,metrics.best_epoch
0,MLPClassifier,64,64,0.0001,Adam,0.0,0.0,0.5,0.0,67.955801,80.918221,24.0
1,MLPClassifier,32,64,0.001,Adam,0.0,0.0,0.0,0.0,65.745856,86.370158,16.0
2,MLPClassifier,64,16,0.001,SGD,0.0,0.0,0.5,0.0,65.193370,69.870875,13.0
3,MLPClassifier,32,16,0.0001,Adam,0.0,0.5,0.0,0.0,65.193370,74.748924,19.0
4,MLPClassifier,64,64,0.001,Adam,0.0,0.5,0.5,0.5,61.325967,68.579627,19.0
5,MLPClassifier,32,64,0.001,SGD,0.2,0.0,0.0,0.0,61.325967,62.553802,46.0
6,MLPClassifier,64,64,0.001,Adam,0.0,0.5,0.5,0.0,60.220994,69.870875,14.0
7,MLPClassifier,32,64,0.0001,Adam,0.2,0.5,0.0,0.5,60.220994,63.701578,24.0
8,MLPClassifier,32,64,0.01,Adam,0.0,0.0,0.5,0.0,59.668508,68.866571,36.0
9,MLPClassifier,32,16,0.001,Adam,0.0,0.5,0.0,0.5,59.116022,63.845050,11.0


In [17]:
hp_cols = ["lr", "batch_size", "dropout", "optimizer", "input_size",
           "HFlip", "VFlip", "RBContrast"]
hp_cols = [c for c in hp_cols if f"params.{c}" in runs.columns]

results = runs[[f"params.{c}" for c in hp_cols] + ["metrics.best_val_accuracy"]].copy()
results.columns = hp_cols + ["best_val_accuracy"]
results["best_val_accuracy"] = pd.to_numeric(results["best_val_accuracy"], errors="coerce")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, hp in enumerate(hp_cols):
    if i >= len(axes):
        break
    grouped = results.groupby(hp)["best_val_accuracy"].mean().sort_index()
    grouped.plot(kind="bar", ax=axes[i], color="steelblue", edgecolor="white")
    axes[i].set_title(f"Val Acc media por {hp}")
    axes[i].set_ylabel("Val Accuracy (%)")
    axes[i].set_xlabel(hp)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle("MLP — Impacto de cada HP en val_accuracy", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("hp_importance_mlp.png", bbox_inches="tight")
plt.show()